# AI 07 - Ev tapşırığı (part 1)
# Maşın öyrənməsi alqoritmləri və reqressiya metrikləri - konspekt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Decision Tree - Qərar ağacı

**İş prinsipi:** Datanı suallarla (məs. `age < 5?`) hissələrə bölür. Hər addımda elə bölgü seçilir ki, xəta (reqressiyada MSE) ən çox azalsın. Sonda yarpaqlarda (leaf) həmin hissənin orta qiyməti proqnoz olur.

**Üstünlükləri:**
- Başa düşmək və vizuallaşdırmaq asandır
- Scaling lazım deyil
- Qeyri-xətti asılılıqları tuta bilir

**Çatışmazlıqları:**
- Asanlıqla overfitting olur (dərin ağac)
- Datada kiçik dəyişiklik ağacı tam dəyişə bilər (qeyri-stabil)

**Tətbiq:** kredit qərarları, xəstəlik diaqnozu, sadə qiymət proqnozu.

# 2. Random Forest - Təsadüfi meşə

**İş prinsipi:** Çoxlu Decision Tree qurulur (bagging). Hər ağac datanın təsadüfi seçilmiş hissəsi (bootstrap) və təsadüfi əlamətlər üzərində öyrənir. Reqressiyada nəticə bütün ağacların proqnozlarının ortasıdır.

**Üstünlükləri:**
- Tək ağacdan daha dəqiq və stabil
- Overfitting-ə az meyllidir
- Feature importance verir

**Çatışmazlıqları:**
- Yavaşdır, çox yaddaş tutur
- Tək ağac kimi asan şərh olunmur

**Tətbiq:** ev/maşın qiyməti, fırıldaqçılığın aşkarlanması, tibb.

# 3. Gradient Boosting - Qradiyent bustinqi

**İş prinsipi:** Ağaclar **ardıcıl** qurulur. Hər yeni kiçik ağac əvvəlki ağacların səhvlərini (residuals) düzəltməyə çalışır. Hər ağacın töhfəsi `learning_rate` ilə kiçildilir. Gradient descent ideyası ilə loss funksiyası minimuma endirilir.

**Üstünlükləri:**
- Çox dəqiq nəticə verir
- Müxtəlif loss funksiyaları ilə işləyir

**Çatışmazlıqları:**
- Ardıcıl olduğu üçün öyrənmə yavaşdır
- Hiperparametrlərə həssasdır, overfitting ola bilər
- Outlier-lərə həssasdır

**Tətbiq:** satış proqnozu, risk qiymətləndirmə, ranking.

# 4. XGBoost - Extreme Gradient Boosting

**İş prinsipi:** Gradient Boosting-in optimallaşdırılmış versiyasıdır. Eyni ideya (ardıcıl ağaclar səhvləri düzəldir), amma daha sürətli və güclü.

**Üstünlükləri:**
- Çox sürətlidir (paralel hesablama)
- Regularization (L1, L2) var - overfitting azalır
- NaN dəyərləri özü idarə edir
- Kaggle yarışlarında çox populyardır

**Çatışmazlıqları:**
- Hiperparametr çoxdur, tənzimləmək çətindir
- Ayrıca kitabxana lazımdır (`pip install xgboost`)

**Tətbiq:** yarışlar, böyük cədvəl datalar, bank, reklam.

# Alqoritmlərin müqayisəsi

| | Decision Tree | Random Forest | Gradient Boosting | XGBoost |
|---|---|---|---|---|
| Ağac sayı | 1 | çox | çox | çox |
| Qurulma | tək | paralel (bagging) | ardıcıl (boosting) | ardıcıl, amma paralel hesablama |
| Overfitting | yüksək | aşağı | orta | aşağı (regularization) |
| Sürət | çox sürətli | orta | yavaş | sürətli |
| Dəqiqlik | aşağı | yaxşı | çox yaxşı | çox yaxşı |

## Gradient Boosting vs XGBoost
- XGBoost-da **regularization** (L1/L2) var, adi GB-də yoxdur.
- XGBoost ağacın qurulmasını **paralel** hesablayır, ona görə daha sürətlidir.
- XGBoost **NaN** dəyərləri özü idarə edir.
- XGBoost-da ikinci dərəcəli törəmə (Hessian) istifadə olunur - optimizasiya daha dəqiqdir.
- XGBoost-da ağacın budanması (pruning), early stopping kimi əlavə imkanlar var.

# Kod nümunəsi

In [ ]:
X, y = make_regression(n_samples=500, n_features=5, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
models = {
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

# xgboost qurulubsa onu da elave edirem
try:
    from xgboost import XGBRegressor
    models['XGBoost'] = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
except ImportError:
    print('xgboost yoxdur: pip install xgboost')

In [ ]:
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    results.append({
        'model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2': r2_score(y_test, y_pred)
    })

pd.DataFrame(results).round(2)

# 2. Reqressiya metrikləri

# MAE - orta mütləq xəta
$$
MAE = \frac{1}{n}\sum_{i=1}^n|y_i - \hat{y_i}|
$$
Orta hesabla proqnoz real qiymətdən nə qədər fərqlənir. Vahidi hədəflə eynidir. 0 - ideal. Bütün xətalar eyni çəkidə sayılır.

# MSE - orta kvadratik xəta
$$
MSE = \frac{1}{n}\sum_{i=1}^n(y_i - \hat{y_i})^2
$$
Xətalar kvadrata yüksəldiyi üçün böyük xətalar çox güclü cəzalandırılır. Vahidi kvadratdır.

# RMSE - MSE-nin kvadrat kökü
$$
RMSE = \sqrt{MSE}
$$
Vahidi hədəflə eynidir, amma böyük xətalara MAE-dən həssasdır.

# $R^2$ - determinasiya əmsalı
$$
R^2 = 1 - \frac{\sum(y_i - \hat{y_i})^2}{\sum(y_i - \bar{y})^2}
$$
1 - model hər şeyi izah edir, 0 - orta qiymət qədər yaxşıdır, < 0 - ortadan da pisdir.

## Sadə hesablama nümunəsi
Real: [10, 20, 30], proqnoz: [12, 18, 33]

Xətalar: 2, 2, 3

- MAE = (2 + 2 + 3) / 3 = 2.33
- MSE = (4 + 4 + 9) / 3 = 5.67
- RMSE = √5.67 = 2.38
- R² = 1 - 17 / 200 = 0.915

In [ ]:
y_true = np.array([10, 20, 30])
y_pred = np.array([12, 18, 33])

print('MAE:', mean_absolute_error(y_true, y_pred))
print('MSE:', mean_squared_error(y_true, y_pred))
print('RMSE:', np.sqrt(mean_squared_error(y_true, y_pred)))
print('R2:', r2_score(y_true, y_pred))

## Böyük xətaya reaksiya
Bir proqnozu çox səhv edək və baxaq hansı metrik nə qədər dəyişir.

In [ ]:
y_pred_big = np.array([12, 18, 50])  # sonuncu xeta 20 oldu

print('MAE:', mean_absolute_error(y_true, y_pred_big))
print('MSE:', mean_squared_error(y_true, y_pred_big))
print('RMSE:', np.sqrt(mean_squared_error(y_true, y_pred_big)))
print('R2:', r2_score(y_true, y_pred_big))

**Nəticə:** bir böyük xəta olanda MAE təxminən 3 dəfə artır, MSE isə ~24 dəfə artır. Yəni MSE və RMSE böyük xətalara çox həssasdır, MAE isə daha "sakit" reaksiya verir. R² də kəskin düşür.